# Financial Tweet Sentiment Classification — Final Pipeline
## Group 42 — Text Mining 2025/2026, NOVA IMS

Single linear flow: load data → preprocess → fine-tune Twitter-RoBERTa
(top two encoder layers + classification head) on the full training set →
predict on the test set → write `pred_42.csv`.

**Runtime**: ~5 minutes on Apple-Silicon MPS, ~25 minutes on CPU.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))
from src import DATA_DIR, MODELS_DIR, OUTPUTS_DIR, RANDOM_STATE
from src.preprocessing import pp_transformer

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print("Environment ready.")


## 1. Load data


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Training: {len(train_df):,} labelled tweets")
print(f"Test:     {len(test_df):,} unlabelled tweets")
print(f"Train label distribution:\n{train_df['label'].value_counts().sort_index().to_dict()}")


## 2. Preprocess
Light cleaning that the Twitter-RoBERTa tokenizer is happy with.


In [ ]:
print("Preprocessing train + test with pp_transformer...")
X_train = np.array([pp_transformer(t) for t in train_df["text"].values])
X_test = np.array([pp_transformer(t) for t in test_df["text"].values])
y_train = train_df["label"].values.astype(int)
test_ids = test_df["id"].values
print(f"Done. Train={len(X_train):,}  Test={len(X_test):,}")


## 3. Load Twitter-RoBERTa, freeze the bottom ten layers
We unfreeze only the top two encoder layers and the classification head
(~14.8M trainable parameters out of 124.6M, roughly 12%).


In [ ]:
CHECKPOINT = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)
model.config.id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

# Freeze everything except the top two encoder layers and the classifier head.
trainable_keep = ("encoder.layer.10", "encoder.layer.11", "classifier")
n_train, n_total = 0, 0
for name, param in model.named_parameters():
    if any(k in name for k in trainable_keep):
        param.requires_grad = True
        n_train += param.numel()
    else:
        param.requires_grad = False
    n_total += param.numel()
print(f"Trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)")


## 4. Tokenise and fine-tune for 4 epochs


In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=False, truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_dict({"text": list(X_train), "label": list(y_train)})
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
# tiny placeholder eval set (Trainer requires one when eval_strategy != 'no')
eval_ds = Dataset.from_dict({"text": list(X_train[:200]), "label": list(y_train[:200])})
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

args = TrainingArguments(
    output_dir=str(MODELS_DIR / "final_finetune_roberta"),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    report_to=[],
    seed=RANDOM_STATE,
    fp16=False,
    bf16=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

trainer.train()


## 5. Predict on the test set


In [ ]:
test_ds = Dataset.from_dict({"text": list(X_test)})
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

preds_out = trainer.predict(test_ds)
test_preds = np.argmax(preds_out.predictions, axis=-1).astype(int)
print(f"Predicted {len(test_preds):,} test rows")
print(f"Distribution: {pd.Series(test_preds).value_counts().sort_index().to_dict()}")


## 6. Write `pred_42.csv`


In [ ]:
out = pd.DataFrame({"id": test_ids, "label": test_preds})
out_path = OUTPUTS_DIR / "pred_42.csv"
out.to_csv(out_path, index=False)
print(f"Saved {len(out):,} predictions to {out_path}")

# Sanity check
assert list(out.columns) == ["id", "label"]
assert out["label"].isin([0, 1, 2]).all()
assert out.isna().sum().sum() == 0
print("OK —", out["label"].value_counts().sort_index().to_dict())
